In [ ]:
import pandas as pd
import numpy as np
from scipy.special import gamma, gammaincc
pd.set_option('display.float_format', '{:.6f}'.format)

# Gamma parameters
gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584
def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den

# =============================================================================
# 0. read data with CR_threshold column added
# =============================================================================
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_with_V_MES.csv')

# =============================================================================
# 1. parameters
# =============================================================================
T = 17
mu_coper = 1.6
mu_mant = 4.2  # improvement factor of MANT
base_year = 2024
        
C_MANT = 48000      # mant action cost
C_COPER = 48000     # coper action cost (假设与mant相同，可调整)
c_factors = [0.0952 + 0.004 * t for t in range(T)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

nb_coper = 7
nb_mant = 36  

# =============================================================================
# 2. alpha extraction for 2023 (baseline) + Volume
# =============================================================================
new_df['csemCur'] = new_df['csemCur'].astype(int)
df_alpha_2023 = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold", "Volume", "Age 2023"]].last().reset_index()
df_alpha_2023.columns = ["csemCur", "alpha_2023", "Pression Nominale Relative", "CR_threshold", "Volume", "Age 2023"]
print("2023 Alpha Values：")
print(df_alpha_2023)
print(f"Count: {len(df_alpha_2023)} csemCur\n")

# # =============================================================================
# # 3. simulation with costs - NEW LOGIC (只有triggered的csem才排序分配任务)
# # =============================================================================
# # 为每个csemCur维护状态
# csem_states = {}
# for _, row in df_alpha_2023.iterrows():
#     csemCur = int(row["csemCur"])
#     csem_states[csemCur] = {
#         'alpha_current': (row["alpha_2023"] / compute_gamma_h(row["Age 2023"], gamma_alpha, gamma_beta)) * compute_gamma_h(row["Age 2023"] + 1, gamma_alpha, gamma_beta),
#         'pression_nominale': row["Pression Nominale Relative"],
#         'cr_threshold': row["CR_threshold"],
#         'volume_csem': row["Volume"],
#         'csem_K': 146 * row["Volume"] / (8.314 * 293.15 *10**3)
#     }

# rows = []
# for t in range(T):
#     year = base_year + t
    
#     # 🔥 步骤1: 找出本年达到criteria的csem
#     triggered_csem = []
#     for csemCur, state in csem_states.items():
#         # 计算泄漏量和interval
#         leak = state['alpha_current'] * 365 * state['csem_K']
#         delta_p = state['pression_nominale'] - state['cr_threshold']
#         interval = 365 / (state['alpha_current'] * 365 / delta_p) if state['alpha_current'] > 0 else np.inf
        
#         # 检查触发条件
#         trigger_this_year = (leak >= 5) or (interval <= 60)
        
#         if trigger_this_year:
#             triggered_csem.append({
#                 'csemCur': csemCur,
#                 'leak': leak,
#                 'interval': interval,
#                 'state': state
#             })
    
#     # 🔥 步骤2: 只对triggered的csem按泄漏量排序
#     triggered_csem.sort(key=lambda x: x['leak'], reverse=True)
    
#     # 🔥 步骤3: 分配任务（只有前nb_mant+nb_coper个triggered csem获得任务）
#     task_assignments = {}
#     for i, csem_info in enumerate(triggered_csem):
#         csemCur = csem_info['csemCur']
#         if i < nb_mant:
#             task_assignments[csemCur] = 'mant'
#         elif i < nb_mant + nb_coper:
#             task_assignments[csemCur] = 'coper'
#         else:
#             task_assignments[csemCur] = None  # triggered但无任务
    
#     # 🔥 步骤4: 处理所有csem（包括未triggered的）
#     for csemCur, state in csem_states.items():
#         # 计算泄漏量和interval
#         leak = state['alpha_current'] * 365 * state['csem_K']
#         delta_p = state['pression_nominale'] - state['cr_threshold']
#         interval = 365 / (state['alpha_current'] * 365 / delta_p) if state['alpha_current'] > 0 else np.inf
        
#         # 检查触发条件
#         trigger_this_year = (leak >= 5) or (interval <= 60)
        
#         # 计算成本
#         leak_cost = leak * C_leak_t[t]
#         task_type = task_assignments.get(csemCur, None)  # 未triggered的csem没有任务
#         action_cost = 0
#         if task_type == 'mant':
#             action_cost = C_MANT
#         elif task_type == 'coper':
#             action_cost = C_COPER
        
#         total_cost = leak_cost + action_cost
        
#         rows.append({
#             "csemCur": csemCur,
#             "year": year,
#             "alpha": state['alpha_current'],
#             "leak": leak,
#             "interval": interval,
#             "triggered": trigger_this_year,
#             "task_type": task_type,
#             "leak_cost": leak_cost,
#             "action_cost": action_cost,
#             "total_cost": total_cost,
#         })
        
#         # 更新下一年alpha
#         if task_type == 'mant':
#             state['alpha_current'] = state['alpha_current'] / mu_mant
#         elif task_type == 'coper':
#             state['alpha_current'] = state['alpha_current'] / mu_coper
#         else:
#             # 无任务则自然增长（包括未triggered和triggered但无任务的）
#             state['alpha_current'] = state['alpha_current'] * 1.01




# 新增修改关于本年达到条件任务应该显示在下一年
# =============================================================================
# 3. ✅ MODIFIED: 状态初始化 + 延迟任务逻辑
# =============================================================================
csem_states = {}
for _, row in df_alpha_2023.iterrows():
    csemCur = int(row["csemCur"])
    csem_states[csemCur] = {
        'alpha_current': (row["alpha_2023"] / compute_gamma_h(row["Age 2023"], gamma_alpha, gamma_beta)) * compute_gamma_h(row["Age 2023"] + 1, gamma_alpha, gamma_beta),
        'pression_nominale': row["Pression Nominale Relative"],
        'cr_threshold': row["CR_threshold"],
        'volume_csem': row["Volume"],
        'csem_K': 146 * row["Volume"] / (8.314 * 293.15 *10**3),
        'pending_task': None  # ✅ 新增：延迟任务状态
    }

rows = []
for t in range(T):
    year = base_year + t
    
    # ✅ NEW1: 每年开头应用上一年pending_task（同参考triggered_previous_year）
    applied_tasks = {}
    for csemCur, state in csem_states.items():
        if state['pending_task']:
            task = state['pending_task']
            if task == 'mant':
                state['alpha_current'] /= mu_mant
            elif task == 'coper':
                state['alpha_current'] /= mu_coper
            applied_tasks[csemCur] = task
            state['pending_task'] = None  # 清空
    
    # 🔥 步骤1-3: 找出triggered，按leak排序，分配任务→存pending（下一年用）
    triggered_csem = []
    for csemCur, state in csem_states.items():
        leak = state['alpha_current'] * 365 * state['csem_K']
        delta_p = state['pression_nominale'] - state['cr_threshold']
        interval = 365 / (state['alpha_current'] * 365 / delta_p) if state['alpha_current'] > 0 else np.inf
        trigger_this_year = (leak >= 5) or (interval <= 60)
        if trigger_this_year:
            triggered_csem.append({'csemCur': csemCur, 'leak': leak, 'interval': interval, 'state': state})
    
    triggered_csem.sort(key=lambda x: x['leak'], reverse=True)
    task_assignments = {}
    for i, csem_info in enumerate(triggered_csem):
        csemCur = csem_info['csemCur']
        if i < nb_mant:
            task_assignments[csemCur] = 'mant'
        elif i < nb_mant + nb_coper:
            task_assignments[csemCur] = 'coper'
    
    # 存入pending（下一年应用）
    for csemCur in task_assignments:
        csem_states[csemCur]['pending_task'] = task_assignments[csemCur]
    
    # 🔥 步骤4: 处理所有csem，task_type/action_cost用本年APPLIED的任务
    for csemCur, state in csem_states.items():
        leak = state['alpha_current'] * 365 * state['csem_K']
        delta_p = state['pression_nominale'] - state['cr_threshold']
        interval = 365 / (state['alpha_current'] * 365 / delta_p) if state['alpha_current'] > 0 else np.inf
        trigger_this_year = (leak >= 5) or (interval <= 60)
        
        leak_cost = leak * C_leak_t[t]
        task_type = applied_tasks.get(csemCur)  # ✅ 本年已应用的任务（上一年决策）
        action_cost = 0
        if task_type == 'mant': action_cost = C_MANT
        elif task_type == 'coper': action_cost = C_COPER
        
        total_cost = leak_cost + action_cost
        
        rows.append({
            "csemCur": csemCur, "year": year, "alpha": state['alpha_current'],
            "leak": leak, "interval": interval, "triggered": trigger_this_year,
            "task_type": task_type,  # ✅ 下一年显示
            "leak_cost": leak_cost, "action_cost": action_cost, "total_cost": total_cost
        })







# =============================================================================
# 4-6. 后续部分 + 🔥 每个csem的所有年详细结果
# =============================================================================
sim_df = pd.DataFrame(rows)
sim_df['csemCur'] = sim_df['csemCur'].astype(int)

print("\n=== Summary Statistics ===")

cost_summary = sim_df.groupby("csemCur")["total_cost"].sum().round(0).reset_index()
cost_summary.columns = ["csemCur", "total_T years_cost"]

leak_summary = sim_df.groupby("csemCur")["leak"].sum().round(3).reset_index()
leak_summary.columns = ["csemCur", "total_leak_kg"]

sim_df['criterion_environment'] = (sim_df['leak'] >= 5)
sim_df['criterion_exploit'] = (sim_df['interval'] <= 60)

trigger_exploit = sim_df.groupby("csemCur")["criterion_exploit"].sum().astype(int).reset_index()
trigger_exploit.columns = ["csemCur", "trigger_exploit_times"]

trigger_env = sim_df.groupby("csemCur")["criterion_environment"].sum().astype(int).reset_index()
trigger_env.columns = ["csemCur", "trigger_env_times"]

# task counts
mant_count = sim_df[sim_df['task_type'] == 'mant'].groupby('csemCur').size().reindex(cost_summary['csemCur'], fill_value=0).reset_index(name='mant_times')
coper_count = sim_df[sim_df['task_type'] == 'coper'].groupby('csemCur').size().reindex(cost_summary['csemCur'], fill_value=0).reset_index(name='coper_times')

total_years_df = sim_df.groupby("csemCur").size().reset_index(name='total_years')

summary_df = (cost_summary.merge(leak_summary, on='csemCur', how='outer')
                       .merge(trigger_exploit, on='csemCur', how='outer')
                       .merge(trigger_env, on='csemCur', how='outer')
                       .merge(total_years_df, on='csemCur', how='outer')
                       .merge(mant_count, on='csemCur', how='outer')
                       .merge(coper_count, on='csemCur', how='outer'))

summary_df['trigger_exploit_rate'] = (summary_df['trigger_exploit_times'] / summary_df['total_years']).round(3)
summary_df['trigger_env_rate'] = (summary_df['trigger_env_times'] / summary_df['total_years']).round(3)
summary_df['total_trigger_rate'] = ((summary_df['trigger_exploit_times'] + summary_df['trigger_env_times']) / summary_df['total_years']).round(3)

print("📊 Overall Summary:")
print(summary_df.round(3))

print("\n📊 Rank by Total Leak (kg):")
print(summary_df.sort_values('total_leak_kg', ascending=False).round(3).head(10))

# detailed results for top 10 csem by total leak
print("\n" + "="*100)
print("🔥 Every CSEM's Detailed Results (Top 10 by Total Leak)")
print("="*100)

# ranking by total leak and showing all years for top 10 csemCur
top_csem = summary_df.sort_values('total_leak_kg', ascending=False).head(10)['csemCur'].tolist()

for csem_id in top_csem:
    print(f"\n📋 CSEM {csem_id} - all years:")
    csem_detail = sim_df[sim_df['csemCur'] == csem_id][['year', 'alpha', 'leak', 'interval', 'triggered', 'task_type', 'leak_cost', 'action_cost', 'total_cost']].round({'alpha':6, 'leak':3, 'leak_cost':0, 'action_cost':0, 'total_cost':0})
    print(csem_detail.to_string(index=False))
    print("-" * 80)

sim_df.to_csv("simulation_with_costs.csv", index=False)
print(f"\n✅ Simulation completed!")
print("✅ Results saved to: simulation_with_costs.csv")
print("✅ Full detailed results available in CSV file")
print("✅ Preview of first 10 rows:")
print(sim_df.head(10).round({'alpha':6, 'leak':3, 'leak_cost':0, 'total_cost':0, 'action_cost':0}))




2023 Alpha Values：
        csemCur  alpha_2023  Pression Nominale Relative  CR_threshold  \
0    5000200795    0.004915                    3.500000      3.199000   
1    5000252448    0.181613                    3.500000      3.199000   
2    5000252449    0.022172                    3.500000      3.199000   
3    5000252450    0.018647                    3.500000      3.199000   
4    5000252524    0.032852                    3.500000      3.199000   
..          ...         ...                         ...           ...   
590  5002343706    0.121390                    3.500000      3.199000   
591  5002343707    0.063426                    3.500000      3.199000   
592  5002699073    0.019229                    3.500000      3.199000   
593  5002903325    0.001703                    3.500000      3.199000   
594  5002903326    0.030425                    3.500000      3.199000   

      Volume  Age 2023  
0   0.757573 39.021918  
1   6.317558 40.024658  
2   6.302540 40.024658  
3   